In [3]:
from time import sleep
from pathlib import Path

from keystone import Ks, KS_ARCH_PPC, KS_MODE_PPC64
ks = Ks(KS_ARCH_PPC, KS_MODE_PPC64)

import helper_funcs as HF

import dolphin_memory_engine as DME
DME.hook()
assert DME.is_hooked(), "Failed to hook to dolphin"
assert DME.read_bytes(0x80000000,6) == b'GZLE01', "Wrong iso, expecting GZLE01"

In [ ]:
rarc1 = Path.cwd() / "model_files" / "iron_boots" / "Vboot.rarc"
rarc2 = Path.cwd() / "model_files" / "iron_boots" / "bleh.rarc"

with open(rarc1, 'rb') as f1, open(rarc2, 'r') as f2:
    bytes1 = f1.read()
    sbytes2 = f2.read()
sbytes1 = bytes1.hex()
N = len(sbytes1)
Nbad = 0
for n in range(N):
    if sbytes1[n] != sbytes2[n]:
        print(n, sbytes1[n-4:n+4], sbytes2[n-4:n+4])
        Nbad += 1
print(Nbad / N)

2104 8d25c10d 8d254018
2105 d25c10da d254018e
2106 25c10dac 254018ed
2107 5c10dace 54018edf
2108 c10dace5 4018edf5
2109 10dace54 018edf54
2110 0dace541 18edf541
2602 16c1287f 16c132ff
2603 6c1287ff 6c132fff
2604 c1287ff3 c132fff7
2607 87ff3414 2fff7414
3610 a8c0ed3a a8c05bbe
3611 8c0ed3a3 8c05bbef
3612 c0ed3a3d c05bbeff
3613 0ed3a3dc 05bbeffc
3614 ed3a3dc1 5bbeffc1
3615 d3a3dc14 bbeffc14
6424 dc5e40e2 dc5ec022
6426 5e40e228 5ec0223e
6428 40e228eb c0223eff
6429 0e228eb4 0223eff4
6430 e228eb41 223eff41
6431 228eb414 23eff414
6697 47740f8e 47741b7e
6698 7740f8e4 7741b7e6
6699 740f8e4d 741b7e6d
6701 0f8e4d8c 1b7e6ddc
6703 8e4d8c12 7e6ddc12
8744 6549bfb3 65493e43
8745 549bfb35 5493e435
8746 49bfb351 493e435b
8749 fb351483 e435bdd3
8750 b351483f 435bdd3f
8751 351483f8 35bdd3f8
8858 7bc1b324 7bc137f5
8859 bc1b324c bc137f5c
8860 c1b324c2 c137f5cf
8861 1b324c2c 137f5cfc
8863 324c2c13 7f5cfc13
9504 c0b2c0fe c0b22c03
9505 0b2c0fee 0b22c030
9506 b2c0feeb b22c0300
9507 2c0feeb6 22c03000
9508 c0feeb

In [8]:
rarc1 = Path.cwd() / "model_files" / "iron_boots" / "Vboot.rarc"
rarc2 = Path.cwd() / "model_files" / "iron_boots" / "bleh.rarc"

with open(rarc1, 'rb') as f1, open(rarc2, 'r') as f2:
    bytes1 = f1.read()
    sbytes2 = f2.read()
sbytes1 = bytes1.hex()
for n in range(len(sbytes1)):
    if sbytes1[n] != sbytes2[n]:
        print(n, sbytes1[n-4:n+4], sbytes2[n-4:n+4])

409 01000000 01002000
415 00000000 0000d000
744 76626f6f 76620000
745 6626f6f7 66200007
746 626f6f74 62000074
747 26f6f742 20000743
750 6f742e62 00743e62
1913 8e040eda 8e04135a
1914 e040edab e04135af
1915 040edab1 04135af5
1917 0edab194 135af5f4
1918 edab1941 35af5f41
1919 dab19413 5af5f413
2104 8d25c10d 8d254018
2105 d25c10da d254018e
2106 25c10dac 254018ed
2107 5c10dace 54018edf
2108 c10dace5 4018edf5
2109 10dace54 018edf54
2110 0dace541 18edf541
2570 9541b8fe 95418bfe
2571 541b8fee 5418bfef
2574 b8feefc1 8bfeffc1
2680 d4a5c0bb d4a54139
2681 4a5c0bb7 4a541397
2682 a5c0bb77 a541397f
2683 5c0bb77b 541397fb
2685 0bb77b64 1397fb74
2687 b77b6419 97fb7419
3562 6ec0c7eb 6ec09deb
3563 ec0c7ebe ec09debe
3567 7ebecc18 debeec18
3976 a8cfc050 a8cf4139
3977 8cfc0502 8cf4139f
3978 cfc05024 cf4139fc
3979 fc05024b f4139fcb
3980 c05024bc 4139fcbc
3981 05024bc4 139fcbc4
5625 47240d80 472411e1
5626 7240d800 72411e10
5627 240d8005 2411e10f
5628 40d8005a 411e10fb
5630 d8005ac1 1e10fbc1
5631 8005ac16 e10f

In [3]:
# Useful constants
PAD1_addr = 0x803F0F34  # controller 1 C/LR data address
PAD2_addr = 0x803F0F3C  # controller 2 C/LR data address
PAD3_addr = 0x803F0F44  # controller 3 C/LR data address
PAD4_addr = 0x803F0F4C  # controller 4 C/LR data address

nop = bytes.fromhex('60000000')         # "no operation" instruction
button_nop = bytes.fromhex('10808080')  # pscmpu1 cr1, p0, p16 (basically a nop; only affects CR1 which nothing should read from. controller inputs are Start + neutral gray stick)

r12 = 0x803F0F3C                        # where the game jumps when ACE is initially triggered (normally set with Link's X+3 position data)

In [38]:
# Custom DME write function designed to simulate USB Gecko transfers
def my_DME_write(addr, word_bytes, pause=0.001, Nreps=1):
    for _ in range(Nreps):   # perform write Nreps times (if worried about DME writes sometimes not going through)
        DME.write_bytes(addr, word_bytes)
        sleep(pause)   # use to simulate USB Gecko's transfer rate (~1kHz)

In [39]:
########################################################################
# Create phase 1 binary file from file of (address, instruction) pairs
########################################################################
phase1_AI_file = "phase1_addr_instruc_pairs.txt"
phase1_bin_file = "phase1.bin"

phase1_AI_pairs = HF.get_addr_value_pairs_from_files(phase1_AI_file, input_type='ASM', output_type='ASM', ks=ks)
HF.phase1_create_bin_file(phase1_AI_pairs, phase1_bin_file, r12=r12, ks=ks)

In [ ]:
#########################################################################################
# Create phase 2 binary file (main payload) from file of (address, hex) pairs
#########################################################################################
#phase2_AI_file = "phase2_addr_instruc_pairs.txt"
payload_folder = Path.cwd() / 'payload_mods'
phase2_file_list = [AH_file for AH_file in payload_folder.iterdir()]
    
#phase2_AH_file = "phase2_addr_hex_pairs.txt"
phase2_bin_file = "phase2.bin"

#phase2_AH_pairs = HF.get_addr_value_pairs_from_file(phase2_AI_file, input_type='ASM', output_type='hex', ks=ks)
#phase2_AH_pairs = HF.get_addr_value_pairs_from_files(phase2_AH_file, input_type='hex', output_type='hex', ks=ks)

HF.phase2_create_bin_from_files(phase2_file_list, phase2_bin_file, input_type='hex', ks=ks)

In [ ]:
######################################################################################################
# PHASE -1 (pre-ACE): Set controllers 2-4 at start (optional, can manually set before the run instead)
######################################################################################################
# nop out controller 2-4 button/left stick data (will be different if using unplug strats; need to test)
for n in range(3):
    button_addr = 0x803F0F38 + n*0x08
    my_DME_write(button_addr, button_nop)

icbi_r12 = bytes.fromhex('7C0067AC')    # icbi r0, r12 ; invalidates instruction cache at r12=0x803F0F3C (pad 2 C/LR address)
b_42 = bytes.fromhex('4BFFFFF0')        # branch backwards 0x10 bytes

my_DME_write(PAD2_addr, nop)       # clear pad 2 C/LR data
my_DME_write(PAD3_addr, icbi_r12)  # invalidate instruction cache at r12=0x803F0F3C so that the CPU sees updates to pad 2 C/LR data
my_DME_write(PAD4_addr, b_42)      # branch from pad 4 -> pad 2; main loop for phase 1

In [42]:
#########################################################
# PHASE 0.5: nop out all non-essential controller data
#########################################################
# nop out all 4 controllers' button/left stick data (pads 3-4 need to already be harmless before this, but might as well ensure their exact values)
for n in range(4):
    button_addr = 0x803F0F30 + n*0x08
    my_DME_write(button_addr, button_nop, Nreps=10)

# nop out controller 1 C-stick/trigger data (could wait until after phase 1, but might as well do it now)    
my_DME_write(PAD1_addr, nop, Nreps=10)

################################################################################
# PHASE 1: Set up input detection & cache management for phase 2 (main payload)
################################################################################
with open(phase1_bin_file, "rb") as f:
    phase1_input_bytes = f.read()

phase1_instruc_list = [phase1_input_bytes[i:i+4] for i in range(0, len(phase1_input_bytes), 4)]
for phase1_instruc in phase1_instruc_list:
    #print(phase1_instruc)
    my_DME_write(PAD2_addr, phase1_instruc, pause=0.001, Nreps=10)
    

In [43]:
###########################################################
# PHASE 1.5: Transition to phase 2 (main payload)
###########################################################
b_cache2 = bytes.fromhex('48000014')        # 0x803F0F3C: b -> 0x803F0F50
my_DME_write(PAD2_addr, b_cache2, Nreps=3)  # have pad 2 C/LR data branch to the phase 2 cache management module so that pad 3-4 updates work (in phase 1, only pad 2's instruction cache gets refreshed)

my_DME_write(PAD3_addr, nop, Nreps=3)       # clear pad 3 C/LR data (icbi r0, r12) to remove phase 1 cache management 
my_DME_write(PAD4_addr, nop, Nreps=3)       # clear pad 4 C/LR data (b->0x803F0F3C) to remove branch to pad 2
my_DME_write(PAD2_addr, nop, Nreps=3)       # clear pad 2 C/LR data

In [44]:
#############################################
# PHASE 2: Write main payload using pads 1-4
#############################################
with open(phase2_bin_file, "rb") as f:
    phase2_input_bytes = f.read()

instructions = [phase2_input_bytes[i:i+4] for i in range(0, len(phase2_input_bytes), 4)]
for n, instruction in enumerate(instructions):
    pad_idx = n % 4
    pad_address = PAD1_addr + (pad_idx * 0x08)  # controller C-stick/LR address
    my_DME_write(pad_address, instruction)

In [45]:
#########################################################################
# PHASE 3: Perform any cleanup (if necessary) and resume gameplay
#########################################################################
# cleanup TBD; could zero out all addresses in phase1_AI_file but doesn't seem necessary

# Branch to safety to resume game
safety_branch4 = bytes.fromhex('4BE24718')   # 0x803F0F4C: b -> 0x80215664
DME.write_bytes(PAD4_addr, safety_branch4)